# Gauss-Newton Cutler-Vallisneri:  truth  ->  0PA MLP

Start at the **true injected parameters** and iterate the CV / Gauss-Newton step

$$\theta_{k+1} = \theta_k + \Gamma(\theta_k)^{-1}\,\langle\,\partial h_{\rm 0PA}(\theta_k)\mid s - h_{\rm 0PA}(\theta_k)\,\rangle$$

with $s = h_{\rm 1PA}(\theta_{\rm tr})$ the injected signal and $h_{\rm 0PA}$ the recovery template.

* The **first step** is exactly the textbook Cutler-Vallisneri bias prediction, $\theta_{\rm tr}+\Delta\theta_{\rm CV}$.
* Iterating should walk from the truth to the 0PA maximum-likelihood point (the MAP), discovered purely by the iteration -- no external best fit is used.
* A **backtracking line search** damps each step (accept only if $\chi^2=\langle s-h\mid s-h\rangle$ decreases), so it cannot diverge from the far-away truth start. Pure CV would take the full step ($\alpha=1$).

Only the 0PA model is considered (no deviations).


In [7]:
import numpy as np

from few.waveform import GenerateEMRIWaveform
from few.waveform.waveform import SuperKludgeWaveform
from fastlisaresponse import ResponseWrapper
from lisatools.detector import EqualArmlengthOrbits
from lisatools.sensitivity import get_sensitivity, A1TDISens, E1TDISens, T1TDISens
from stableemrifisher.utils import generate_PSD, inner_product, fishinv
from stableemrifisher.fisher import StableEMRIFisher

try:
    import cupy as cp
    xp = cp
except ImportError:
    xp = np
    print("[INFO] CuPy not found, using NumPy instead.")

F_MIN = 1e-5


def _to_float(x):
    return float(x.get()) if hasattr(x, "get") else float(x)


def make_freq_mask(n, dt, fmin):
    return (xp.fft.rfftfreq(n, dt) > fmin)[1:]


def highpass_clip(waveform, dt, fmin):
    n = waveform.shape[-1]
    freq = xp.fft.rfftfreq(n, dt)
    return xp.fft.irfft(xp.fft.rfft(waveform, axis=-1) * (freq >= fmin), n=n, axis=-1)


In [8]:
# ======================================================================
# CASE: EMRI idx 0    (swap this whole block for idx 9 / 13)
# ======================================================================
signal_param = {
    "m1": 1000000.0, "m2": 10.0, "a": 0.9, "p0": 7.5, "e0": 0.5,
    "xI0": 1.0, "dist": 5.0,
    "qS": 0.7853981633974483, "phiS": 1.0,
    "qK": 1.0, "phiK": 1.0471975511965976,
    "Phi_phi0": 0.9, "Phi_theta0": 0.5, "Phi_r0": 0.4,
    "dev_1": 0.0, "dev_2": 0.0,
}
dt, T, chi2 = 5.0, 1.0, 0.0          # chi2 = secondary-spin waveform parameter

use_gpu = True
nchannels = 3
param_names_14 = ["m1", "m2", "a", "p0", "e0", "xI0", "dist", "qS", "phiS",
                  "qK", "phiK", "Phi_phi0", "Phi_theta0", "Phi_r0"]
params_to_infer = ["m1", "m2", "a", "p0", "e0", "qS", "phiS", "Phi_phi0", "Phi_r0"]
true_inferred = np.array([signal_param[n] for n in params_to_infer])


In [9]:
t0 = 10000.0
channels = [A1TDISens, E1TDISens, T1TDISens][:nchannels]
tdi_chan = {2: "AE", 3: "AET"}[nchannels]
noise_kwargs = [{"sens_fn": ch} for ch in channels]


def build_response_kwargs():
    return dict(
        Tobs=T, t0=t0, dt=dt, index_lambda=8, index_beta=7,
        flip_hx=True, is_ecliptic_latitude=False, remove_garbage="zero",
        orbits=EqualArmlengthOrbits(use_gpu=use_gpu),
        force_backend="cuda12x" if use_gpu else "cpu",
        order=20, tdi="1st generation", tdi_chan=tdi_chan,
    )


waveform_model = GenerateEMRIWaveform(
    SuperKludgeWaveform, sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=False, use_gpu=use_gpu,
)
waveform_response = ResponseWrapper(waveform_gen=waveform_model, **build_response_kwargs())


def response_params(inferred_dict, evolve_1pa):
    """Full response-wrapper argument list; non-inferred params held at the truth."""
    p14 = {n: signal_param[n] for n in param_names_14}
    p14.update(inferred_dict)
    dev_1 = inferred_dict.get("dev_1", signal_param["dev_1"])
    dev_2 = inferred_dict.get("dev_2", signal_param["dev_2"])
    return [p14[n] for n in param_names_14] + [
        chi2, evolve_1pa, False, False, False, dev_1, dev_2,
    ]


def make_0pa(inferred_dict):
    """0PA response template at the given inferred point (cheap: no derivatives)."""
    return xp.array(
        waveform_response(*response_params(inferred_dict, evolve_1pa=False))
    )[:nchannels, :]


# --- 1PA injected signal at the truth ---
waveform_true = xp.array(
    waveform_response(*response_params(dict(zip(params_to_infer, true_inferred)),
                                       evolve_1pa=True))
)[:nchannels, :]
waveform_true = highpass_clip(waveform_true, dt, F_MIN)

PSD_funcs = xp.array(generate_PSD(
    waveform=waveform_true, dt=dt, noise_PSD=get_sensitivity,
    channels=channels, noise_kwargs=noise_kwargs, use_gpu=use_gpu,
))
freq_mask = make_freq_mask(waveform_true.shape[-1], dt, F_MIN)


def _ip(a, b):
    return _to_float(inner_product(a, b, PSD=PSD_funcs, dt=dt,
                                   freq_mask=freq_mask, use_gpu=use_gpu))


def normalized_overlap(a, b):
    return _ip(a, b) / np.sqrt(_ip(a, a) * _ip(b, b))


def residual_chi2(h):
    r = waveform_true - h
    return _ip(r, r)


snr = np.sqrt(_ip(waveform_true, waveform_true))
print(f"[SIGNAL] 1PA SNR = {snr:.4f}   N = {waveform_true.shape[-1]}")
print(f"[CHECK ] overlap 0PA(truth) vs 1PA = {normalized_overlap(waveform_true, make_0pa(dict(zip(params_to_infer, true_inferred)))):.6f}")


[SIGNAL] 1PA SNR = 44.6145   N = 6311629
[CHECK ] overlap 0PA(truth) vs 1PA = 0.112384


In [10]:
sef = StableEMRIFisher(
    waveform_class=SuperKludgeWaveform,
    waveform_class_kwargs=dict(sum_kwargs=dict(pad_output=True, odd_len=True)),
    waveform_generator=GenerateEMRIWaveform,
    waveform_generator_kwargs=dict(return_list=False),
    ResponseWrapper=ResponseWrapper,
    ResponseWrapper_kwargs=build_response_kwargs(),
    stats_for_nerds=True, use_gpu=use_gpu, deriv_type="stable",
    noise_model=get_sensitivity, noise_kwargs=noise_kwargs, channels=channels,
    T=T, dt=dt, stability_plot=False, der_order=6, Ndelta=12,
    plunge_check=True, return_derivatives=True,
)

add_param_args = {
    "chi2": chi2, "evolve_1PA": False, "evolve_primary": False,
    "evolve_2PA": False, "deviation_included": False, "dev_1": 0.0, "dev_2": 0.0,
}


def sef_fisher_derivs(inferred_vec, deltas):
    """0PA Fisher matrix (NumPy) and derivatives (CuPy) at the given inferred point."""
    wp = {n: signal_param[n] for n in param_names_14}
    wp.update(dict(zip(params_to_infer, inferred_vec)))
    F = sef(
        wave_params={n: wp[n] for n in param_names_14},
        param_names=params_to_infer, add_param_args=add_param_args,
        deltas=deltas, live_dangerously=False, stability_plot=False,
        der_order=8, Ndelta=(20 if deltas is None else None),
    )
    return np.asarray(F[-1], dtype=float), xp.array(F[0])


In [11]:
# --- Levenberg-Marquardt controls --------------------------------------
LAMBDA0   = 1e-3      # initial damping
LAMBDA_UP = 10.0      # multiply lambda when a trial step fails (-> smaller, safer)
LAMBDA_DN = 0.1       # multiply lambda when a step succeeds     (-> more Gauss-Newton)
MAX_ITERS = 60
MAX_INNER = 25        # max lambda-increases per outer iteration
CONV_TOL_SIGMA = 1e-2
# -----------------------------------------------------------------------
# Solve  (Gamma + lambda * diag(Gamma)) delta = <dh | s - h>.
# lambda -> 0 is pure Gauss-Newton / CV; lambda large is short gradient-descent.
# The diag(Gamma) scaling (Marquardt) handles the huge EMRI dynamic range and
# also regularizes the non-positive-definite Fisher.

npar = len(params_to_infer)
current = true_inferred.astype(float).copy()      # START AT THE TRUTH
stable_deltas = None
cv_first_step = None
lam = LAMBDA0

print(f"{'it':>3} {'lambda':>9} {'overlap':>12} {'chi2':>12} {'|step/sig|':>11} {'n_lam':>6}")
for it in range(MAX_ITERS):
    G, dH = sef_fisher_derivs(current, stable_deltas)
    if stable_deltas is None:
        stable_deltas = sef.deltas

    h_ap = make_0pa(dict(zip(params_to_infer, current)))
    r = waveform_true - h_ap
    g = np.array([_ip(dH[j], r) for j in range(npar)])          # <d_j h | s - h>
    sigma = np.sqrt(np.abs(np.diag(fishinv(current[0], G, index_of_M=0))))
    chi2_cur = residual_chi2(h_ap)
    ov_cur = normalized_overlap(waveform_true, h_ap)
    if cv_first_step is None:
        cv_first_step = np.linalg.solve(G, g)                   # pure GN = textbook CV

    # LM inner loop: raise lambda until a damped step actually lowers chi2
    diagG = np.diag(np.diag(G))
    delta, accepted, n_lam = np.zeros(npar), False, 0
    for _ in range(MAX_INNER):
        try:
            delta = np.linalg.solve(G + lam * diagG, g)
        except np.linalg.LinAlgError:
            lam *= LAMBDA_UP; n_lam += 1; continue
        if residual_chi2(make_0pa(dict(zip(params_to_infer, current + delta)))) < chi2_cur:
            accepted = True
            break
        lam *= LAMBDA_UP; n_lam += 1

    step_sig = float(np.max(np.abs(delta / sigma)))
    print(f"{it:>3} {lam:>9.1e} {ov_cur:>12.8f} {chi2_cur:>12.4e} {step_sig:>11.3e} {n_lam:>6d}")

    if not accepted:
        print("LM: no chi2 decrease even at large lambda -> stop")
        break
    current = current + delta
    lam = max(lam * LAMBDA_DN, 1e-14)                           # relax after success
    if step_sig < CONV_TOL_SIGMA:
        print("converged: |step/sigma| < tol")
        break


 it    lambda      overlap         chi2  |step/sig|  n_lam
Body is not plunging, Fisher should be stable.
waveform shape: (3, 6311629)
wave ndim: 2
Computing SNR for parameters: (np.float64(1000000.0), np.float64(10.0), np.float64(0.9), np.float64(7.5), np.float64(0.5), 1.0, 5.0, np.float64(0.7853981633974483), np.float64(1.0), 1.0, 1.0471975511965976, np.float64(0.9), 0.5, np.float64(0.4), 0.0, False, False, False, False, 0.0, 0.0)
Waveform Generated. SNR: 44.58510859200293
calculating stable deltas...
Gamma_ii for m1: 1817.8983629070692
Gamma_ii for m1: 1817.898362902699
Gamma_ii for m1: 1817.8983629034556
Gamma_ii for m1: 1817.8983628784667
Gamma_ii for m1: 1817.8983628700562
Gamma_ii for m1: 1817.8983628498406
Gamma_ii for m1: 1817.8983630284486
Gamma_ii for m1: 1817.8983626761174
Gamma_ii for m1: 1817.8983632703937
Gamma_ii for m1: 1817.8983627752273
Gamma_ii for m1: 1817.898362307066
Gamma_ii for m1: 1817.89836538773
Gamma_ii for m1: 1817.8983601385557
Gamma_ii for m1: 1817.89836

In [12]:
# --- the point the iteration climbed to from the truth: the discovered 0PA MAP ---
Gf, _ = sef_fisher_derivs(current, stable_deltas)
sigma_f = np.sqrt(np.abs(np.diag(fishinv(current[0], Gf, index_of_M=0))))

print("discovered 0PA MAP (climbed from the truth) -- bias = MAP - truth")
print(f"{'param':10s} {'true':>18} {'0PA MAP':>18} {'MAP-true':>13} {'(MAP-true)/sig':>15}")
for name, tv, cv, sg in zip(params_to_infer, true_inferred, current, sigma_f):
    print(f"{name:10s} {tv:>18.8e} {cv:>18.8e} {cv - tv:>13.3e} {(cv - tv) / sg:>15.3f}")

ov_map = normalized_overlap(waveform_true, make_0pa(dict(zip(params_to_infer, current))))
print(f"\noverlap 0PA(MAP) vs 1PA = {ov_map:.8f}")

print("\none-step CV bias prediction (first iteration only):  theta_tr + delta_CV")
for name, tv, d in zip(params_to_infer, true_inferred, cv_first_step):
    print(f"  {name:10s} true={tv: .8e}  +delta_CV={d: .3e}  -> {tv + d: .8e}")


Body is not plunging, Fisher should be stable.
waveform shape: (3, 6311629)
wave ndim: 2
Computing SNR for parameters: (np.float64(1000148.8585885885), np.float64(10.001052611500734), np.float64(0.9000964863635473), np.float64(7.499425407190435), np.float64(0.49997465683634684), 1.0, 5.0, np.float64(0.783363425610641), np.float64(0.9996242901573592), 1.0, 1.0471975511965976, np.float64(0.9093361604679479), 0.5, np.float64(0.39611228365608786), 0.0, False, False, False, False, 0.0, 0.0)
Waveform Generated. SNR: 44.65838189014368
calculating Fisher matrix...
Finished derivatives
Calculated Fisher is *atleast* positive-definite.
Time taken to compute FM is 2.9049956798553467 seconds
discovered 0PA MAP (climbed from the truth) -- bias = MAP - truth
param                    true            0PA MAP      MAP-true  (MAP-true)/sig
m1             1.00000000e+06     1.00014886e+06     1.489e+02           0.576
m2             1.00000000e+01     1.00010526e+01     1.053e-03           0.576
a       